In [ ]:
import pandas as pd
import re
import pymorphy3
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF, TruncatedSVD
from gensim import corpora
from gensim.models import LdaModel, Nmf

import warnings
warnings.filterwarnings('ignore')

In [49]:
train_df = pd.read_parquet(r'..\lab_5\train.parquet', engine='fastparquet')[:2000]
test_df = pd.read_parquet(r'..\lab_5\test.parquet', engine='fastparquet')

train_df

,text,summary,topic,url,title,date
0,Сладострастник в течение трех лет преследовал ...,Старший преподаватель института коммунального ...,incident,https://www.mk.ru/incident/article/2010/01/05/...,Педофил преследовал подростка три года,06/01/2010
1,Буквально за час до боя курантов в подземном п...,Манежная площадь Москвы стала местом последнег...,incident,https://www.mk.ru/incident/article/2010/01/05/...,Таджики устроили резню на Манежной площади под...,06/01/2010
2,"Там они покатались на лыжах и снегоходах, пооб...",Президент РФ Дмитрий Медведев с семьей проводи...,politics,https://www.mk.ru/politics/article/2010/01/05/...,"""""""За все платит президент""""""",06/01/2010
3,Сосед расстрелял соседа из-за претензий по пов...,Первое убийство в 2010 году произошло в Москве...,incident,https://www.mk.ru/incident/article/2010/01/05/...,Первое убийство года спровоцировал потоп,06/01/2010
4,Причиной трагедии специалисты считают нарушени...,"Шесть человек, в том числе 9-летний ребенок, з...",incident,https://www.mk.ru/incident/article/2010/01/05/...,Целая семья не пережила газовой атаки,06/01/2010
...,...,...,...,...,...,...
1995,"Скачка закончилась, а безусловный победитель т...",Как итоги выборов в туманном Альбионе могут от...,politics,https://www.mk.ru/politics/article/2010/05/07/...,Британия на весах,08/05/2010
1996,.Сводный оркестр — поистине белая кость военно...,Главный военный дирижер Валерий Халилов раскры...,politics,https://www.mk.ru/politics/interview/2010/05/0...,Победа в темпе марша. АУДИО,08/05/2010
1997,Эту горькую правду знает весь Междуреченск. Зд...,"Массовая гибель шахтеров не несчастный случай,...",incident,https://www.mk.ru/incident/article/2010/05/10/...,Распад начался не с “Распадской”. ВИДЕО,11/05/2010
1998,8 мая президент Медведев поставил свою подпись...,О последствиях реформы бюджетной сферы не знаю...,politics,https://www.mk.ru/politics/article/2010/05/10/...,В России вводят сырой закон,11/05/2010


In [50]:
def preprocess_text(text, morph_analyzer, stop_words, min_word_len=3):
    if not isinstance(text, str) or not text.strip():
        return ""
    
    text = text.lower()
    text = re.sub(r'[^a-zа-яё\s]', ' ', text)
    tokens = word_tokenize(text)
    
    filtered_tokens = []
    for token in tokens:
        if len(token) < min_word_len or token in stop_words:
            continue
        
        parsed = morph_analyzer.parse(token)[0]
        normal_form = parsed.normal_form
        if len(normal_form) >= min_word_len and normal_form not in stop_words:
            filtered_tokens.append(normal_form)
    
    return ' '.join(filtered_tokens)


morph = pymorphy3.MorphAnalyzer()
stop_words = set(stopwords.words('russian'))

train_df['processed_text'] = train_df['text'].apply(
    lambda x: preprocess_text(x, morph, stop_words, min_word_len=3)
)
test_df['processed_text'] = test_df['text'].apply(
    lambda x: preprocess_text(x, morph, stop_words, min_word_len=3)
)

train_df = train_df[train_df['processed_text'].str.len() > 0]
test_df = test_df[test_df['processed_text'].str.len() > 0]

train_df

,text,summary,topic,url,title,date,processed_text
0,Сладострастник в течение трех лет преследовал ...,Старший преподаватель института коммунального ...,incident,https://www.mk.ru/incident/article/2010/01/05/...,Педофил преследовал подростка три года,06/01/2010,сладострастник течение год преследовать подрос...
1,Буквально за час до боя курантов в подземном п...,Манежная площадь Москвы стала местом последнег...,incident,https://www.mk.ru/incident/article/2010/01/05/...,Таджики устроили резню на Манежной площади под...,06/01/2010,буквально час бой курант подземный переход ста...
2,"Там они покатались на лыжах и снегоходах, пооб...",Президент РФ Дмитрий Медведев с семьей проводи...,politics,https://www.mk.ru/politics/article/2010/01/05/...,"""""""За все платит президент""""""",06/01/2010,покататься лыжа снегоход пообщаться отдыхать п...
3,Сосед расстрелял соседа из-за претензий по пов...,Первое убийство в 2010 году произошло в Москве...,incident,https://www.mk.ru/incident/article/2010/01/05/...,Первое убийство года спровоцировал потоп,06/01/2010,сосед расстрелять сосед претензия повод затопл...
4,Причиной трагедии специалисты считают нарушени...,"Шесть человек, в том числе 9-летний ребенок, з...",incident,https://www.mk.ru/incident/article/2010/01/05/...,Целая семья не пережила газовой атаки,06/01/2010,причина трагедия специалист считать нарушение ...
...,...,...,...,...,...,...,...
1995,"Скачка закончилась, а безусловный победитель т...",Как итоги выборов в туманном Альбионе могут от...,politics,https://www.mk.ru/politics/article/2010/05/07/...,Британия на весах,08/05/2010,скачок закончиться безусловный победитель выяв...
1996,.Сводный оркестр — поистине белая кость военно...,Главный военный дирижер Валерий Халилов раскры...,politics,https://www.mk.ru/politics/interview/2010/05/0...,Победа в темпе марша. АУДИО,08/05/2010,сводный оркестр поистине белый кость военный д...
1997,Эту горькую правду знает весь Междуреченск. Зд...,"Массовая гибель шахтеров не несчастный случай,...",incident,https://www.mk.ru/incident/article/2010/05/10/...,Распад начался не с “Распадской”. ВИДЕО,11/05/2010,горький правда знать весь междуреченск вообще ...
1998,8 мая президент Медведев поставил свою подпись...,О последствиях реформы бюджетной сферы не знаю...,politics,https://www.mk.ru/politics/article/2010/05/10/...,В России вводят сырой закон,11/05/2010,май президент медведев поставить подпись закон...


In [52]:
vectorizer = CountVectorizer(
    max_features=15000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.9
)

train_matrix = vectorizer.fit_transform(train_df['processed_text'])
test_matrix = vectorizer.transform(test_df['processed_text'])
vocabulary = vectorizer.get_feature_names_out()

print(f"Размер словаря: {len(vocabulary)}")
print(f"Размер матрицы обучающих данных: {train_matrix.shape}")

Размер словаря: 15000
Размер матрицы обучающих данных: (2000, 15000)


In [53]:
num_topics = 50

# **sklearn**

In [54]:
def print_topics(model, feature_names, n_top_words=10):
    for topic_idx, topic in enumerate(model.components_):
        message = f"Topic {topic_idx}: "
        message += " ".join([feature_names[i]
                           for i in topic.argsort()[:-n_top_words - 1:-1]])
        print(message)

## LDA

In [55]:
sklearn_lda = LatentDirichletAllocation(
    n_components=num_topics,
    max_iter=20,
    batch_size=1000,
    random_state=42,
    n_jobs=-1
)

sklearn_lda.fit(train_matrix)

,n_components,50
,doc_topic_prior,None
,topic_word_prior,None
,learning_method,'batch'
,learning_decay,0.7
,learning_offset,10.0
,max_iter,20
,batch_size,1000
,evaluate_every,-1
,total_samples,1000000.0
,perp_tol,0.1


In [56]:
print("Темы sklearn LDA:")
print_topics(sklearn_lda, vocabulary)

Темы sklearn LDA:
Topic 0: это который новый школа оружие год ребёнок стать травматический время
Topic 1: дело уголовный преступление следователь сотрудник суд убийство следствие который милиция
Topic 2: рубль деньга фирма сотрудник тыс задержать сумма тыс рубль мвд сообщить
Topic 3: матч команда игрок футбол клуб лига россия чемпион первый спартак
Topic 4: война победа военный советский армия ветеран солдат великий наш день
Topic 5: который это стать солдат сказать президент часть день речник первый
Topic 6: машина водитель милиционер стать авто около произойти место который летний
Topic 7: ленин егэ язык год который июнь кремль человек белый зелёный
Topic 8: год россия новый страна это который день градус эксперимент система
Topic 9: москва московский область год город московский область место власть городской мэр
Topic 10: который сотрудник преступник рубль около казино грабитель бандит находиться место
Topic 11: пенсия год инвалид выплата размер пенсионный компенсация гражданин фон

## NMF

In [60]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.9
)

train_tfidf = tfidf_vectorizer.fit_transform(train_df['processed_text'])
tfidf_vocab = tfidf_vectorizer.get_feature_names_out()

sklearn_nmf = NMF(
    n_components=num_topics,
    init='nndsvda',
    max_iter=500,
    random_state=42
)

sklearn_nmf.fit(train_tfidf)

,n_components,50
,init,'nndsvda'
,solver,'cd'
,beta_loss,'frobenius'
,tol,0.0001
,max_iter,500
,random_state,42
,alpha_W,0.0
,alpha_H,'same'
,l1_ratio,0.0
,verbose,0


In [61]:
print("Темы sklearn NMF:")
print_topics(sklearn_nmf, tfidf_vocab)

Темы sklearn NMF:
Topic 0: это человек очень говорить жизнь знать сказать ваш мочь наш
Topic 1: преступник сыщик задержать грабитель кража преступный жертва улица деньга преступление
Topic 2: год прошлый год прошлый родиться март загс смертность сравнение год год год родиться
Topic 3: олимпийский ванкувер игра медаль олимпиада спортсмен сочи тренер наш плющенко
Topic 4: ребёнок родитель мальчик усыновление астахов детский семья приёмный право ребёнок уполномоченный
Topic 5: врач пациент медик больной помощь лечение больница операция отделение клиника
Topic 6: область московский область московский подмосковье муниципальный муниципальный образование бюджет население регион образование
Topic 7: солдат военный парад войско офицер армия площадь минобороны знамя красный
Topic 8: россия страна наш экономика российский это экономический кризис власть государство
Topic 9: война ветеран победа сталин великий великий отечественный отечественный война отечественный участник советский
Topic 10: мат

## SVD/LSA

In [62]:
sklearn_svd = TruncatedSVD(
    n_components=num_topics,
    algorithm='randomized',
    n_iter=10,
    random_state=42
)

sklearn_svd.fit(train_tfidf)

,n_components,50
,algorithm,'randomized'
,n_iter,10
,n_oversamples,10
,power_iteration_normalizer,'auto'
,random_state,42
,tol,0.0


In [63]:
print("Темы sklearn SVD/LSA:")
print_topics(sklearn_svd, tfidf_vocab)

Темы sklearn SVD/LSA:
Topic 0: год это который человек свой стать наш ребёнок время россия
Topic 1: это наш россия команда игра сборная мир очень тренер матч
Topic 2: год рубль руб область тыс цена правительство власть средство московский область
Topic 3: ребёнок это фильм человек жизнь очень жить родитель мама говорить
Topic 4: ребёнок школа родитель мальчик семья врач девочка мать год малыш
Topic 5: суд год дело рубль деньга тыс уголовный задержать уголовный дело тыс рубль
Topic 6: суд ребёнок дело это судья водитель адвокат должный сотрудник документ
Topic 7: ребёнок президент школа янукович украина война россия родитель власть тимошенко
Topic 8: янукович россия президент украина страна пациент тимошенко выборы медик врач
Topic 9: суд квартира война судебный судья военный победа ветеран верховный суд верховный
Topic 10: матч квартира руб игрок янукович команда суд дом лига украина
Topic 11: квартира дом олимпийский ванкувер спорт спортсмен суд олимпиада жильё медаль
Topic 12: школа 

# **gensim**

In [64]:
tokenized_docs = [doc.split() for doc in train_df['processed_text']]
dictionary = corpora.Dictionary(tokenized_docs)
dictionary.filter_extremes(no_below=5, no_above=0.9)
corpus = [dictionary.doc2bow(doc) for doc in tokenized_docs]

## LDA

In [65]:
gensim_lda = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=num_topics,
    random_state=42,
    passes=10,
    per_word_topics=True
)

In [67]:
print("Темы gensim LDA:")
for topic_id in range(num_topics):
    print(f"Topic {topic_id}: ", end='')
    topic_words = gensim_lda.show_topic(topic_id, topn=10)
    print(" ".join([word for word, _ in topic_words]))

Темы gensim LDA:
Topic 0: год виза режим паспорт документ должный россия служба евросоюз отдел
Topic 1: стать волк известно флажок дело время год слово человек снегоход
Topic 2: магазин стать человек который год сайт порядок каждый продавщица спиртное
Topic 3: закон школа который это год свой время новый стать сми
Topic 4: год врач пациент операция который больной лечение россия медицинский человек
Topic 5: ребёнок год это который дело свой родитель человек время суд
Topic 6: янукович год москва школа витя кремль река время который первый
Topic 7: россия год который это экономика страна наш бизнес власть российский
Topic 8: год который президент свой это дело украина россия стать янукович
Topic 9: год война который ветеран область победа стать московский москва дом
Topic 10: ирина захаркина это девочка момент ребёнок очень даша адвокат подсудимый
Topic 11: пенсия год пенсионный который часть страховой фонд взнос размер новый
Topic 12: сталин знамя алексей генерал лейтенант комната свой

## NMF

In [69]:
gensim_nmf = Nmf(
    corpus=corpus,
    id2word=dictionary,
    num_topics=num_topics,
    random_state=42,
    passes=10
)

In [70]:
print("Темы gensim NMF:")
for topic_id in range(num_topics):
    print(f"Topic {topic_id}: ", end='')
    topic_words = gensim_nmf.show_topic(topic_id, topn=10)
    print(" ".join([word for word, _ in topic_words]))

Темы gensim NMF:
Topic 0: мир наш чемпионат сборная турнир команда чемпион тренер игрок памятник
Topic 1: улица переулок площадь набережная проезд проспект больший шоссе большой малый
Topic 2: который это игра россия свой олимпийский говорить первый сборная сказать
Topic 3: ребёнок врач серёжа клиника галина орган лечение месяц германия мама
Topic 4: который свой это закон власть человек мафия наш дело российский
Topic 5: год преступление республика работа орган вклад инвалид группа сотрудник также
Topic 6: президент украина выборы тимошенко голос партия тур политический который кандидат
Topic 7: который работа город должный оружие это год область спорт закон
Topic 8: трасса спортсмен спорт спортивный летний олимпиада дорога олимпийский ванкувер лыжный
Topic 9: млн год руб тыс президент супруг доход дом квартира заработать
Topic 10: ребёнок год который родитель семья ребята это говорить свой жить
Topic 11: наш год область страна это война московский подмосковье победа россия
Topic 12: 